In [9]:
import sys
print(sys.version)

3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]


# Budget vs Actual Analytics

## Project Overview

This project analyzes an organization's budgeted and actual expenses to identify budget overruns, major cost drivers, and areas requiring stronger cost control.

The analysis follows an end-to-end data analytics workflow using Python, SQL, and Power BI. Python is used for data quality assessment, cleaning, validation, and exploratory analysis before the cleaned data is analyzed further using SQL and visualized in Power BI.

## Business Objective

The objective is to compare budgeted expenses with actual spending and identify:

- Overall budget performance
- Departments with significant overspending
- Expense categories contributing to unfavorable variance
- Department-category combinations driving cost overruns
- Monthly and annual spending trends
- Regional areas with higher budget variance
- Payment methods associated with spending patterns
- Specific areas that should be prioritized for cost-control actions

## Tools & Technologies

- Python
- Pandas
- NumPy
- Jupyter Notebook
- MySQL
- Power BI
- DAX
- Microsoft Excel

## Dataset

The dataset contains financial transactions recorded from 2021 to 2023, including budget amounts, actual spending, departments, expense categories, regions, payment methods, and transaction identifiers.

The dataset contains intentionally introduced data-quality issues such as missing values and duplicate records, which are addressed during the data-cleaning stage.

## Analytical Approach

The project follows these stages:

1. Data loading and initial inspection
2. Data quality assessment
3. Missing-value analysis
4. Duplicate-record investigation
5. Data cleaning
6. Financial validation
7. Variance calculation
8. Exploratory business analysis
9. SQL-based analysis
10. Power BI dashboard development
11. Business insights and recommendations

## 1. Import Libraries & Load Dataset

Pandas and NumPy are imported for data manipulation, analysis, and numerical operations.

The raw financial dataset is then loaded from the project's `Data/Raw` directory using Pandas. The first five records are displayed to verify that the dataset has been loaded correctly.

In [2]:
import pandas as pd
import numpy as np

file_path = "../Data/Raw/Budget_vs_Actual_Data.xlsx"

df = pd.read_excel(file_path)

df.head()

,Date,Department,Category,Region,Budget Amount,Actual Amount,Payment Method,Transaction ID
0,2023-05-11,Sales,Travel,North,126096,43048,Card,TXN100000
1,2023-11-11,Marketing,Salaries,East,19702,87896,Bank Transfer,TXN100001
2,2021-05-02,IT,Training,Central,108523,103632,Card,TXN100002
3,2022-04-12,Marketing,Salaries,North,114711,105574,Cash,TXN100003
4,2021-11-27,Sales,Utilities,Central,121895,64314,UPI,TXN100004


### Initial Dataset Preview

The initial preview confirms that the dataset contains transaction-level financial records, including dates, departments, expense categories, regions, budgeted amounts, actual amounts, payment methods, and transaction IDs.

## 2. Initial Dataset Inspection

Before cleaning the data, the dataset is inspected to understand its structure, dimensions, column names, data types, and basic statistical characteristics.

This step helps identify potential data-quality issues before performing any transformations.

In [3]:
# Basic dataset information

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Dataset Shape: (10010, 8)

Column Names:
['Date', 'Department', 'Category', 'Region', 'Budget Amount', 'Actual Amount', 'Payment Method', 'Transaction ID']

Data Types:
Date              datetime64[us]
Department                   str
Category                     str
Region                       str
Budget Amount              int64
Actual Amount              int64
Payment Method               str
Transaction ID               str
dtype: object


### Dataset Structure

The dataset contains **10,010 records and 8 columns** before cleaning.

The fields include transaction dates, organizational departments, expense categories, geographical regions, budgeted amounts, actual amounts, payment methods, and transaction identifiers.

The data types are appropriate for the initial analysis, with the date field stored as a datetime type and financial amounts stored as numeric values.

## 3. Missing Value Analysis

Missing values are checked to identify incomplete records that may affect the accuracy of the analysis.

The number of missing values in each column is examined before applying any cleaning or imputation techniques.

In [4]:
# Check missing values and duplicates

print("Missing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Missing Values:
Date              0
Department        1
Category          1
Region            3
Budget Amount     0
Actual Amount     0
Payment Method    0
Transaction ID    3
dtype: int64

Duplicate Rows: 10


### Missing Value Findings

The initial analysis identified **8 missing values** across the dataset.

Missing values were found in the Department, Category, Region, and Transaction ID fields. These records will be investigated further before the cleaning stage to determine the appropriate treatment for each field.

## 4. Descriptive Statistics

Descriptive statistics are used to examine the numerical financial fields, including their average values, minimum and maximum amounts, and overall distribution.

This provides an initial understanding of the budget and actual spending values before performing detailed financial analysis.

In [5]:
# Statistical summary of numerical columns

df.describe()

,Date,Budget Amount,Actual Amount
count,10010,10010.000000,10010.000000
mean,2022-07-08 14:11:29.070929,79531.584316,89031.239261
min,2021-01-01 00:00:00,10005.000000,8002.000000
25%,2021-10-08 00:00:00,44379.000000,48570.250000
50%,2022-07-12 00:00:00,79515.500000,89188.000000
75%,2023-04-09 00:00:00,114575.250000,130586.000000
max,2023-12-31 00:00:00,149992.000000,169973.000000
std,NaN,40490.798172,46947.492216


### Initial Financial Overview

The descriptive statistics provide an initial view of the range and distribution of budgeted and actual transaction amounts.

The financial fields contain valid positive values, which allows the analysis to proceed to more detailed data-quality and financial validation checks.

## 5. Duplicate Record Investigation

The dataset initially contained 10,010 records. Duplicate analysis identified 20 duplicate rows representing 10 duplicated transactions. The duplicated records contained identical values across all columns, including Transaction ID. These were classified as true duplicate records and will be removed during the cleaning stage.

In [6]:
# Show duplicate records

duplicates = df[df.duplicated(keep=False)]

duplicates.sort_values(by="Transaction ID")

,Date,Department,Category,Region,Budget Amount,Actual Amount,Payment Method,Transaction ID
1933,2022-01-03,Finance,Infrastructure,North,68546,91058,Cash,TXN101933
10008,2022-01-03,Finance,Infrastructure,North,68546,91058,Cash,TXN101933
10007,2021-08-23,Sales,Utilities,East,82957,72156,UPI,TXN101989
1989,2021-08-23,Sales,Utilities,East,82957,72156,UPI,TXN101989
10006,2021-05-12,Sales,Infrastructure,South,28919,65335,Card,TXN102041
2041,2021-05-12,Sales,Infrastructure,South,28919,65335,Card,TXN102041
3850,2021-11-24,Marketing,Utilities,South,69911,157260,Cash,TXN103850
10001,2021-11-24,Marketing,Utilities,South,69911,157260,Cash,TXN103850
3886,2023-10-14,Marketing,Salaries,East,30855,36986,Card,TXN103886
10003,2023-10-14,Marketing,Salaries,East,30855,36986,Card,TXN103886


In [7]:
print("Number of duplicate records:", len(duplicates))

Number of duplicate records: 20


### Duplicate Record Findings

The duplicate analysis identified **20 duplicate rows**, representing **10 duplicated transactions**.

The duplicated records contain identical values across all columns, including the Transaction ID. Therefore, these records are classified as true duplicates rather than legitimate repeated transactions.

They will be removed during the data-cleaning stage to prevent duplicate transactions from affecting the financial analysis.

## 6. Missing Value Investigation

After identifying missing values, the affected records are investigated individually to understand where the missing information occurs and whether the records can be retained.

This step helps determine an appropriate treatment for each missing field while avoiding unnecessary removal of valid financial transactions.

In [8]:
# Investigate missing values

missing_records = df[df.isnull().any(axis=1)]

missing_records

,Date,Department,Category,Region,Budget Amount,Actual Amount,Payment Method,Transaction ID
1337,2022-01-30,Finance,Travel,NaN,112971,152398,UPI,TXN101337
2659,2023-01-07,NaN,Infrastructure,Central,115763,114627,Bank Transfer,TXN102659
2735,2023-11-04,Operations,NaN,North,40145,15085,Cash,TXN102735
3011,2023-04-04,IT,Marketing,South,27025,48529,Card,NaN
5088,2022-05-16,IT,Training,NaN,12940,93544,UPI,TXN105088
5307,2023-02-19,Operations,Salaries,East,17385,79153,UPI,NaN
6196,2023-08-06,Sales,Utilities,East,89127,153343,Card,NaN
9356,2021-06-09,IT,Salaries,NaN,74723,137717,Card,TXN109356


In [11]:
print("Records with missing values:", len(missing_records))

Records with missing values: 8


## Missing Value Investigation

The dataset contained 8 records with missing values. Missing values were found in Department, Category, Region, and Transaction ID.

Because the affected records contain valid financial amounts and represent only a very small portion of the dataset, the records will be retained rather than deleted.

Missing categorical values will be labeled as "Unknown". Missing Transaction IDs will receive clearly identifiable placeholder IDs to preserve transaction-level records without fabricating original identifiers.

In [9]:
# Create a copy of the raw dataset
df_clean = df.copy()

# Remove the 10 true duplicate rows
df_clean = df_clean.drop_duplicates()

# Fill missing categorical values
for col in ["Department", "Category", "Region"]:
    df_clean[col] = df_clean[col].fillna("Unknown")

# Create placeholder IDs for missing Transaction IDs
missing_id = df_clean["Transaction ID"].isna()

df_clean.loc[missing_id, "Transaction ID"] = (
    "MISSING_ID_" + df_clean.loc[missing_id].index.astype(str)
)

# Check the results
print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))

Original rows: 10010
Cleaned rows: 10000
Rows removed: 10


In [10]:
# Validate the cleaned dataset

print("Missing values:")
print(df_clean.isnull().sum())

print("\nDuplicate rows:", df_clean.duplicated().sum())

print("\nData shape:", df_clean.shape)

Missing values:
Date              0
Department        0
Category          0
Region            0
Budget Amount     0
Actual Amount     0
Payment Method    0
Transaction ID    0
dtype: int64

Duplicate rows: 0

Data shape: (10000, 8)


### Missing Value Findings

The investigation identified **8 records containing missing values**.

The missing values occur in the following fields:

- **Department:** 1 missing value
- **Category:** 1 missing value
- **Region:** 3 missing values
- **Transaction ID:** 3 missing values

The affected records contain valid budget and actual amounts, so they should not be removed solely because of missing categorical or transaction-identification fields.

Categorical missing values will be labeled as **"Unknown"**, while missing Transaction IDs will be assigned unique placeholder identifiers during the cleaning stage.

## 7. Financial Data Validation

Before performing financial analysis, the budget and actual amount fields are validated to identify invalid financial values.

The validation checks for:

- Minimum and maximum values
- Zero values
- Negative values

This ensures that the financial amounts are within a reasonable range and that no invalid negative or zero transactions are included in the analysis.

In [11]:
# Financial data validation

print("Budget Amount:")
print("Minimum:", df_clean["Budget Amount"].min())
print("Maximum:", df_clean["Budget Amount"].max())
print("Zero values:", (df_clean["Budget Amount"] == 0).sum())
print("Negative values:", (df_clean["Budget Amount"] < 0).sum())

print("\nActual Amount:")
print("Minimum:", df_clean["Actual Amount"].min())
print("Maximum:", df_clean["Actual Amount"].max())
print("Zero values:", (df_clean["Actual Amount"] == 0).sum())
print("Negative values:", (df_clean["Actual Amount"] < 0).sum())

Budget Amount:
Minimum: 10005
Maximum: 149992
Zero values: 0
Negative values: 0

Actual Amount:
Minimum: 8002
Maximum: 169973
Zero values: 0
Negative values: 0


### Financial Validation Findings

The financial validation confirms that both Budget Amount and Actual Amount contain valid positive values.

No zero or negative financial transactions were identified. Budget amounts range from 10,005 to 149,992, while actual amounts range from 8,002 to 169,973.

Therefore, no additional corrections are required for the financial amount fields, and the dataset is suitable for variance and spending analysis.

## 8. Budget vs Actual Validation

The relationship between actual spending and budgeted spending is examined to determine how frequently transactions exceed their allocated budgets.

A transaction is considered **over budget** when the Actual Amount is greater than the Budget Amount. This check provides an early indication of the extent of budget overruns in the dataset.

In [12]:
# Check whether Actual Amount exceeds Budget Amount

df_clean["Variance"] = (
    df_clean["Actual Amount"] - df_clean["Budget Amount"]
)

print("Transactions over budget:", (df_clean["Variance"] > 0).sum())
print("Transactions under budget:", (df_clean["Variance"] < 0).sum())
print("Transactions exactly on budget:", (df_clean["Variance"] == 0).sum())

Transactions over budget: 5590
Transactions under budget: 4410
Transactions exactly on budget: 0


### Budget Overrun Findings

The validation shows that **5,590 transactions (55.9%)** exceeded their allocated budget, while **4,410 transactions (44.1%)** remained at or below budget.

This indicates that budget overruns are relatively widespread across the dataset rather than being limited to a small number of transactions. Further analysis is therefore required to identify the departments, categories, regions, and periods contributing most to the unfavorable variance.

## 9. Overall Financial Performance

The overall financial performance is evaluated by comparing total budgeted amounts with total actual spending.

The analysis calculates total budget, total actual spending, absolute variance, and variance percentage. These metrics provide an executive-level view of the organization's overall budget performance.

A positive variance indicates that actual spending exceeded the allocated budget, representing unfavorable budget performance.

In [13]:
# Overall financial performance

total_budget = df_clean["Budget Amount"].sum()
total_actual = df_clean["Actual Amount"].sum()
total_variance = total_actual - total_budget
variance_pct = (total_variance / total_budget) * 100

print("Total Budget:", total_budget)
print("Total Actual Spending:", total_actual)
print("Total Variance:", total_variance)
print("Overall Variance %:", round(variance_pct, 2), "%")

Total Budget: 795469256
Total Actual Spending: 890267029
Total Variance: 94797773
Overall Variance %: 11.92 %


### Overall Financial Findings

The analysis shows total budgeted spending of **795,469,256** compared with actual spending of **890,267,029**.

This resulted in an unfavorable variance of **94,797,773**, meaning actual spending exceeded the overall budget.

The overall variance percentage is **11.92%**, indicating that actual spending was approximately 11.92% higher than the allocated budget.

This establishes a significant overall budget-control issue and provides the baseline for identifying the departments, categories, regions, and time periods responsible for the overspending.

## 10. Average Transaction Values

Average transaction values are calculated to understand the typical budgeted amount, actual spending amount, and variance per transaction.

These metrics complement the overall financial totals by showing the average financial impact at the individual transaction level.

In [14]:
# Average transaction values

print("Average Budget:", round(df_clean["Budget Amount"].mean(), 2))
print("Average Actual Spending:", round(df_clean["Actual Amount"].mean(), 2))
print("Average Variance:", round(df_clean["Variance"].mean(), 2))

Average Budget: 79546.93
Average Actual Spending: 89026.7
Average Variance: 9479.78


### Average Transaction Findings

The average budgeted amount per transaction is approximately **79,546.93**, while the average actual spending is approximately **89,026.70**.

This represents an average unfavorable variance of approximately **9,479.78 per transaction**.

The difference between average budget and actual spending is consistent with the overall finding that actual spending exceeded the allocated budget across the dataset.

## 11. Department-Level Financial Performance

Financial performance is analyzed at the department level to identify departments with the highest spending and the largest unfavorable budget variances.

For each department, total budget, total actual spending, transaction count, absolute variance, and variance percentage are calculated.

This analysis helps identify departments that require greater cost-control attention.

In [15]:
# Department-level financial performance

department_analysis = (
    df_clean
    .groupby("Department")
    .agg(
        Total_Budget=("Budget Amount", "sum"),
        Total_Actual=("Actual Amount", "sum"),
        Transactions=("Transaction ID", "count")
    )
    .reset_index()
)

department_analysis["Variance"] = (
    department_analysis["Total_Actual"]
    - department_analysis["Total_Budget"]
)

department_analysis["Variance_%"] = (
    department_analysis["Variance"]
    / department_analysis["Total_Budget"]
    * 100
)

department_analysis.sort_values(
    "Variance",
    ascending=False
)

,Department,Total_Budget,Total_Actual,Transactions,Variance,Variance_%
3,Marketing,131939479,151816098,1696,19876619,15.064952
1,HR,137371163,155439763,1727,18068600,13.153124
5,Sales,134702488,151344999,1685,16642511,12.355014
4,Operations,132622213,147803987,1649,15181774,11.447384
2,IT,125771001,139663026,1589,13892025,11.045491
0,Finance,132947149,144084529,1653,11137380,8.377299
6,Unknown,115763,114627,1,-1136,-0.981315


### Department-Level Findings

Marketing recorded the **largest unfavorable variance**, with actual spending exceeding its budget by **19,876,619**, representing a variance of approximately **15.06%**.

HR had the second-largest dollar variance at **18,068,600**, followed by Sales at **16,642,511**.

Marketing also recorded the highest variance percentage among the major departments, indicating that it requires particular attention from a cost-control perspective.

All major departments recorded unfavorable variances, suggesting that budget overruns are not isolated to a single department.

## 12. Category-Level Financial Performance

Financial performance is analyzed across expense categories to identify which types of expenses contribute most to budget overruns.

For each category, total budget, total actual spending, transaction count, absolute variance, and variance percentage are calculated.

This analysis helps identify expense categories that may require cost optimization or closer budget monitoring.

In [16]:
# Category-level financial performance

category_analysis = (
    df_clean
    .groupby("Category")
    .agg(
        Total_Budget=("Budget Amount", "sum"),
        Total_Actual=("Actual Amount", "sum"),
        Transactions=("Transaction ID", "count")
    )
    .reset_index()
)

category_analysis["Variance"] = (
    category_analysis["Total_Actual"]
    - category_analysis["Total_Budget"]
)

category_analysis["Variance_%"] = (
    category_analysis["Variance"]
    / category_analysis["Total_Budget"]
    * 100
)

category_analysis.sort_values(
    "Variance",
    ascending=False
)

,Category,Total_Budget,Total_Actual,Transactions,Variance,Variance_%
2,Salaries,127131160,147744247,1632,20613087,16.214032
6,Utilities,137891618,155260320,1743,17368702,12.595908
1,Marketing,130224132,146525696,1622,16301564,12.518082
4,Travel,139968858,154885333,1742,14916475,10.656996
0,Infrastructure,127813870,140825422,1623,13011552,10.180078
3,Training,132399473,145010926,1637,12611453,9.525305
5,Unknown,40145,15085,1,-25060,-62.423714


### Category-Level Findings

Salaries recorded the **largest unfavorable variance**, with actual spending exceeding the budget by **20,613,087**, representing a variance of approximately **16.21%**.

Utilities had the second-largest dollar variance at **17,368,702**, followed by Marketing at **16,301,564**.

Salaries also recorded the highest variance percentage among the major expense categories, indicating that it represents a significant cost-control area.

All major expense categories recorded unfavorable variances, suggesting that budget overruns are distributed across multiple types of expenses rather than being concentrated in a single category.

## 13. Department × Category Analysis

To identify more specific cost drivers, financial performance is analyzed across combinations of departments and expense categories.

This analysis compares budgeted and actual spending for each department-category combination and calculates the resulting dollar variance and variance percentage.

The results help identify specific areas where budget overruns are concentrated and provide a stronger basis for targeted cost-control recommendations.

In [17]:
# Department × Category analysis

dept_category = (
    df_clean
    .groupby(["Department", "Category"])
    .agg(
        Total_Budget=("Budget Amount", "sum"),
        Total_Actual=("Actual Amount", "sum"),
        Transactions=("Transaction ID", "count")
    )
    .reset_index()
)

dept_category["Variance"] = (
    dept_category["Total_Actual"]
    - dept_category["Total_Budget"]
)

dept_category["Variance_%"] = (
    dept_category["Variance"]
    / dept_category["Total_Budget"]
    * 100
)

dept_category.sort_values(
    "Variance",
    ascending=False
).head(10)

,Department,Category,Total_Budget,Total_Actual,Transactions,Variance,Variance_%
21,Marketing,Training,21294820,26796378,280,5501558,25.835194
22,Marketing,Travel,20727724,24862371,274,4134647,19.947424
26,Operations,Salaries,19686711,23803114,258,4116403,20.909552
36,Sales,Utilities,22308763,26387064,288,4078301,18.281162
30,Operations,Utilities,23738950,27727006,305,3988056,16.799631
14,IT,Salaries,19840742,23815157,265,3974415,20.031585
9,HR,Training,23139073,26838701,295,3699628,15.988661
25,Operations,Marketing,20366640,23988226,261,3621586,17.781951
19,Marketing,Marketing,23587963,27159617,297,3571654,15.141850
20,Marketing,Salaries,20971783,24476425,273,3504642,16.711226


### Department × Category Findings

The Department × Category analysis provides a more detailed view of the organization's budget overruns.

The **Marketing–Training** combination recorded the largest unfavorable dollar variance of approximately **5,501,558**, with actual spending exceeding its budget by approximately **25.84%**.

Marketing–Travel and Operations–Salaries were the next major contributors, with variances of approximately **4,134,647** and **4,116,403**, respectively.

Several combinations also recorded variance percentages above 20%, indicating that some specific department-category areas experienced substantial budget overruns.

These results suggest that cost-control efforts should focus on specific combinations rather than applying the same corrective action across entire departments or expense categories.

## 14. Monthly Financial Performance

Monthly financial performance is analyzed to identify trends in budget and actual spending over time.

The analysis compares monthly budgeted amounts with actual spending and calculates the resulting dollar and percentage variance.

This helps determine whether budget overruns are isolated to specific periods, recurring throughout the year, or persistent across the full analysis period.

In [18]:
monthly_analysis = (
    df_clean
    .groupby(df_clean["Date"].dt.to_period("M"))
    .agg(
        Total_Budget=("Budget Amount", "sum"),
        Total_Actual=("Actual Amount", "sum")
    )
    .reset_index()
)

monthly_analysis["Variance"] = (
    monthly_analysis["Total_Actual"]
    - monthly_analysis["Total_Budget"]
)

monthly_analysis["Variance_%"] = (
    monthly_analysis["Variance"]
    / monthly_analysis["Total_Budget"]
    * 100
)

monthly_analysis

,Date,Total_Budget,Total_Actual,Variance,Variance_%
0,2021-01,21587408,24913213,3325805,15.406227
1,2021-02,18142623,20632263,2489640,13.722602
2,2021-03,19747700,24295909,4548209,23.031588
3,2021-04,19710246,24364104,4653858,23.611364
4,2021-05,24597175,27250381,2653206,10.786629
5,2021-06,22265011,22396228,131217,0.589342
6,2021-07,21471995,24685975,3213980,14.968241
7,2021-08,23607929,25003789,1395860,5.912675
8,2021-09,22494061,25829986,3335925,14.830248
9,2021-10,19594167,22180032,2585865,13.197116


### Monthly Financial Findings

The monthly analysis shows that actual spending exceeded the allocated budget throughout the analyzed period.

The highest monthly variance percentages occurred in **April 2021 (23.61%)**, **March 2021 (23.03%)**, and **December 2022 (22.55%)**.

The lowest variance percentage occurred in **June 2021 (0.59%)**, although spending still remained above budget.

The persistence of unfavorable variance across the monthly periods suggests that budget overruns are not limited to isolated months and may reflect broader structural cost-control issues.

## 15. Region × Department Analysis

Financial performance is analyzed across regions and departments to identify geographical areas where specific departments are experiencing higher budget overruns.

The analysis compares budgeted and actual spending for each region-department combination and calculates the resulting dollar and percentage variance.

This helps identify regional cost-control hotspots and supports more targeted management decisions.

In [19]:
region_department = (
    df_clean
    .groupby(["Region", "Department"])
    .agg(
        Total_Budget=("Budget Amount", "sum"),
        Total_Actual=("Actual Amount", "sum"),
        Transactions=("Transaction ID", "count")
    )
    .reset_index()
)

region_department["Variance"] = (
    region_department["Total_Actual"]
    - region_department["Total_Budget"]
)

region_department["Variance_%"] = (
    region_department["Variance"]
    / region_department["Total_Budget"]
    * 100
)

region_department.sort_values(
    "Variance",
    ascending=False
).head(10)

,Region,Department,Total_Budget,Total_Actual,Transactions,Variance,Variance_%
10,East,Marketing,24997181,30776969,336,5779788,23.121759
8,East,HR,27783470,33502840,354,5719370,20.585514
1,Central,HR,26356973,30453248,342,4096275,15.541523
32,West,Sales,26421944,30501431,337,4079487,15.439769
17,North,Operations,26588569,30633422,335,4044853,15.212752
3,Central,Marketing,27226138,31028434,351,3802296,13.965609
31,West,Operations,24432588,28189953,311,3757365,15.378498
18,North,Sales,25206194,28882464,317,3676270,14.584788
19,South,Finance,25273646,28921898,321,3648252,14.435005
30,West,Marketing,27414144,31008233,344,3594089,13.110346


### Region × Department Findings

The Region × Department analysis identifies specific geographical and departmental combinations contributing to budget overruns.

The **East–Marketing** combination recorded the largest unfavorable variance at approximately **5,779,788**, with actual spending exceeding its budget by approximately **23.12%**.

The **East–HR** combination was another significant cost-control area, with an unfavorable variance of approximately **5,719,370** and a variance percentage of **20.59%**.

Other notable combinations include North–Operations, West–Sales, West–Operations, and Central–HR.

The results indicate that some budget-control issues are concentrated within specific regional and departmental combinations rather than being evenly distributed across the organization.

## 16. Annual Financial Performance

Annual financial performance is analyzed to compare budgeted and actual spending across the three years covered by the dataset.

The analysis calculates total budget, total actual spending, transaction count, dollar variance, and variance percentage for each year.

This helps identify which year experienced the greatest budget pressure and whether financial performance is improving or worsening over time.

In [20]:
annual_analysis = (
    df_clean
    .groupby(df_clean["Date"].dt.year)
    .agg(
        Total_Budget=("Budget Amount", "sum"),
        Total_Actual=("Actual Amount", "sum"),
        Transaction_Count=("Transaction ID", "count"),
        Total_Variance=("Variance", "sum")
    )
    .reset_index()
    .rename(columns={"Date": "Year"})
)

annual_analysis["Variance_Percentage"] = (
    annual_analysis["Total_Variance"]
    / annual_analysis["Total_Budget"]
    * 100
)

annual_analysis

,Year,Total_Budget,Total_Actual,Transaction_Count,Total_Variance,Variance_Percentage
0,2021,256681020,287511086,3216,30830066,12.011042
1,2022,266434536,302869150,3399,36434614,13.674884
2,2023,272353700,299886793,3385,27533093,10.109315


### Annual Financial Findings

The annual analysis shows that all three years recorded unfavorable budget variances.

**2022** experienced the highest dollar variance of approximately **36,434,614** and the highest variance percentage at **13.67%**, making it the most financially challenging year.

In **2023**, the variance decreased to approximately **27,533,093**, while the variance percentage improved to **10.11%**.

Although financial performance improved in 2023, actual spending remained above budget. This indicates progress in cost control but also suggests that the underlying budget-overrun issue has not been fully resolved.

## 17. Regional Financial Performance

Financial performance is analyzed across regions to identify geographical areas with higher spending and unfavorable budget variance.

For each region, total budget, total actual spending, transaction count, dollar variance, and variance percentage are calculated.

This analysis helps identify regional cost-control hotspots and determine whether budget overruns are concentrated in particular geographical areas.

In [21]:
regional_analysis = (
    df_clean[df_clean["Region"] != "Unknown"]
    .groupby("Region")
    .agg(
        Total_Budget=("Budget Amount", "sum"),
        Total_Actual=("Actual Amount", "sum"),
        Transaction_Count=("Transaction ID", "count"),
        Total_Variance=("Variance", "sum")
    )
    .reset_index()
)

regional_analysis["Variance_Percentage"] = (
    regional_analysis["Total_Variance"]
    / regional_analysis["Total_Budget"]
    * 100
)

regional_analysis = regional_analysis.sort_values(
    "Total_Variance",
    ascending=False
)

regional_analysis

,Region,Total_Budget,Total_Actual,Transaction_Count,Total_Variance,Variance_Percentage
1,East,159685797,181076974,2038,21391177,13.395792
0,Central,162657334,182078152,2044,19420818,11.939712
3,South,157991439,176760699,1996,18769260,11.879922
4,West,157004504,175505843,1971,18501339,11.783954
2,North,157929548,174461702,1948,16532154,10.468056


### Regional Financial Findings

The **East region** recorded the largest unfavorable dollar variance of approximately **21,391,177** and also had the highest variance percentage at **13.40%**.

Central recorded the second-largest dollar variance at approximately **19,420,818**, while South and West recorded similar levels of unfavorable variance.

North had the lowest variance among the regions at approximately **16,532,154**, but it still exceeded its allocated budget.

All five regions recorded unfavorable variances, indicating that budget overruns are widespread geographically rather than being isolated to one region.

## 18. Payment Method Analysis

Financial performance is analyzed across payment methods to determine whether spending patterns and budget overruns vary significantly by payment channel.

For each payment method, total budget, total actual spending, transaction count, dollar variance, and variance percentage are calculated.

This analysis helps determine whether payment method is a meaningful contributor to the overall budget variance.

In [22]:
payment_analysis = (
    df_clean
    .groupby("Payment Method")
    .agg(
        Total_Budget=("Budget Amount", "sum"),
        Total_Actual=("Actual Amount", "sum"),
        Transaction_Count=("Transaction ID", "count"),
        Total_Variance=("Variance", "sum")
    )
    .reset_index()
)

payment_analysis["Variance_Percentage"] = (
    payment_analysis["Total_Variance"]
    / payment_analysis["Total_Budget"]
    * 100
)

payment_analysis = payment_analysis.sort_values(
    "Total_Variance",
    ascending=False
)

payment_analysis

,Payment Method,Total_Budget,Total_Actual,Transaction_Count,Total_Variance,Variance_Percentage
0,Bank Transfer,196236754,221246962,2464,25010208,12.744915
1,Card,198450529,222698430,2495,24247901,12.218612
3,UPI,204534322,228697575,2554,24163253,11.813789
2,Cash,196247651,217624062,2487,21376411,10.892569


### Payment Method Findings

Bank Transfer recorded the largest unfavorable dollar variance of approximately **25,010,208**, followed by Card at approximately **24,247,901**.

However, the variance percentages across all payment methods are relatively close, ranging from approximately **10.89% to 12.74%**.

This suggests that payment method is unlikely to be a primary driver of the organization's budget overruns. The more significant cost-control opportunities are found at the department, category, regional, and department-category levels.

## 19. Management Priorities

The final analysis identifies the specific department, category, and regional combinations with the largest unfavorable budget variances.

These combinations represent the highest-priority areas for management attention and provide a practical basis for targeted cost-control actions.

The analysis is ranked by total dollar variance to prioritize areas with the greatest financial impact.

In [23]:
management_priorities = (
    df_clean[
        (df_clean["Department"] != "Unknown") &
        (df_clean["Category"] != "Unknown") &
        (df_clean["Region"] != "Unknown")
    ]
    .groupby(["Department", "Category", "Region"])
    .agg(
        Total_Budget=("Budget Amount", "sum"),
        Total_Actual=("Actual Amount", "sum"),
        Total_Variance=("Variance", "sum"),
        Transaction_Count=("Transaction ID", "count")
    )
    .reset_index()
)

management_priorities["Variance_Percentage"] = (
    management_priorities["Total_Variance"]
    / management_priorities["Total_Budget"]
    * 100
)

management_priorities = management_priorities.sort_values(
    "Total_Variance",
    ascending=False
)

management_priorities.head(10)

,Department,Category,Region,Total_Budget,Total_Actual,Total_Variance,Transaction_Count,Variance_Percentage
106,Marketing,Training,East,4791687,6907712,2116025,67,44.160334
105,Marketing,Training,Central,3827057,5524866,1697809,55,44.363306
171,Sales,Travel,East,3984240,5597725,1613485,59,40.496682
56,HR,Utilities,East,4471917,5927580,1455663,57,32.551208
54,HR,Travel,West,4910691,6302830,1392139,67,28.349147
70,IT,Salaries,Central,3195926,4577665,1381739,51,43.234387
157,Sales,Marketing,North,4823463,6148376,1324913,64,27.468087
96,Marketing,Marketing,East,3481394,4801331,1319937,50,37.914037
126,Operations,Marketing,East,3292264,4560872,1268608,49,38.532997
111,Marketing,Travel,East,4036676,5302057,1265381,56,31.347103


### Management Priority Findings

The management-priority analysis identifies the combinations with the largest financial impact.

The **East–Marketing–Training** combination recorded the largest dollar variance at approximately **2,116,025**, with actual spending exceeding its budget by approximately **44.16%**.

Other high-priority areas include Marketing–Training in Central, Sales–Travel in East, HR–Utilities in East, and IT–Salaries in Central.

The results highlight a recurring concentration of cost overruns in the **East region**, particularly within Marketing and related expense categories.

These findings can be used to prioritize detailed budget reviews, investigate the underlying causes of overspending, and develop targeted cost-control measures.

# 20. Overall Findings & Business Recommendations

## Key Findings

- Total actual spending exceeded the total budget by approximately **94.80 million**, resulting in an overall unfavorable variance of **11.92%**.
- **55.9% of transactions** exceeded their allocated budget.
- **Marketing** recorded the largest departmental variance at approximately **19.88 million** and a variance percentage of **15.06%**.
- **Salaries** recorded the largest category-level variance at approximately **20.61 million**, with a variance percentage of **16.21%**.
- The **Marketing–Training** combination recorded the largest department-category variance at approximately **5.50 million**.
- **2022** recorded the highest annual variance at approximately **36.43 million**, with a variance percentage of **13.67%**.
- The **East region** recorded the largest regional variance at approximately **21.39 million** and the highest regional variance percentage of **13.40%**.
- Payment-method variances were relatively similar, suggesting that payment method is not a primary driver of overall overspending.
- The persistence of unfavorable variance across months and regions suggests that budget overruns are a broader cost-control issue rather than isolated events.

## Business Recommendations

### 1. Review Marketing and Training Expenses
Management should investigate the drivers behind high Marketing and Training expenses, particularly in the East region.

### 2. Strengthen Salary Budget Planning
Salary-related spending should be reviewed against workforce plans and departmental budgets to identify recurring gaps between planned and actual costs.

### 3. Focus on High-Variance Combinations
Rather than applying broad cost reductions, management should prioritize specific department-category-region combinations with consistently high unfavorable variance.

### 4. Improve Budget Forecasting
Historical spending patterns should be incorporated into future budgets, particularly for departments and categories that repeatedly exceed their allocations.

### 5. Monitor Regional Cost Performance
The East region should receive additional budget monitoring because it recorded the highest regional variance.

### 6. Establish Ongoing Variance Monitoring
Monthly budget-versus-actual monitoring should be implemented to identify unfavorable trends earlier and allow corrective action before variances become significant.

## Conclusion

The analysis indicates that budget overruns are widespread across the organization, with specific departments, categories, and regional combinations contributing disproportionately to the overall variance.

The findings provide a foundation for targeted cost-control strategies and will be further explored through SQL analysis and an interactive Power BI dashboard.

In [24]:
cleaned_path = "../Data/Cleaned/Budget_vs_Actual_Cleaned.csv"

df_clean.to_csv(cleaned_path, index=False)

print("Cleaned dataset saved successfully.")
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))

Cleaned dataset saved successfully.
Rows: 10000
Columns: 9
